# Content-based recommender system
## Asian Restaurant Chain (Cyprus)
### This notebook uses the *AsianRestaurantChain_Cyprus_2018* dataset

Content-based recommendations systems recommend items that are similar to items previously purchased by other customers. The models used in content-based recommender systems identify common features across customer preferences based on the historical purchase data of the customer (e.g., what items did customers who made similar purchases also made).
<br>In this example, there are two objectives concerning the sales to "Delivery" customers:
- Develop a system to answer the "Did you forget problem" for systems with recursive purchasing. For example, if in previous orders, every time a customer order Chicken Sour he/she also ordered Wine, and in the current order, Chicken Sour is present, but not Wine, recommend Wine.
- Develop a system to recommend products that similar customers purchase. For example, based on previous purchases from all customers, recommend a drink to go together with a specific dish.

This is a simple example. Not all CRISP-DM phases are described in high detail.


(c) Nuno António, João Silveira 2023 - Rev. 1.11

### Dataset description

#### This is real data. Some quality issues may arise. Please, do not share the data outside this course.

Each row in the dataset represents a document line (invoice line). The dataset is composed of the following columns:
- **DocNumber**: number of the document. The document number repeats in as many rows as the rows in the original document (invoice)
- **ProductDesignation**: product designation
- **ProductFamily**: name of the family of the product. A product can only be member of one only family
- **Qty**: quantity
- **TotalAmount**: sale price of the total quantity
- **InvoiceDateHour**: date and hour when the document was issued
- **EmployeeID**: ID of the employee who issued the document
- **IsDelivery**: indication if sale was a delivery or a dine-inn (1: delivery, 0: dine-inn)
- **Pax**: number of persons at the table
- **CustomerID**: ID of the customer (if a customer record was assigned to the sale)
- **CustomerCity**: city of the customer (usually only employed in delivery)
- **CustomerSince**: date of creation of the customer

## Initial setup and modules/packages loading

In [1]:
# Packages
import os
import numpy as np
import pandas as pd
import re

In [2]:
# Distances functions
from sklearn.metrics import pairwise_distances

# For more info on how to implement in Python check https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html#examples-using-sklearn-metrics-pairwise-distances and https://docs.scipy.org/doc/scipy/reference/spatial.distance.html

In [3]:
# Global definitions
baseFolder = os.getcwd()
exportsFolder = baseFolder + os.sep +'Exports' + os.sep

## Data understanding

In [4]:
# Load data 
ds = pd.read_csv('AsianRestaurantChain_Cyprus_2018.csv', sep=";")

In [5]:
# Summary statistics for all variables
ds.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
DocNumber,84109,11147,TK0110053522018,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductDesignation,84109,255,MINERAL WATER 1.5LT,7061,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductFamily,84109,27,STARTERS,14148,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Qty,84109.0,NaN,NaN,NaN,1.261256,0.92378,1.0,1.0,1.0,1.0,53.0
TotalAmount,84109.0,NaN,NaN,NaN,9.83142,20.656789,0.0,3.0,6.9,12.6,3000.0
InvoiceDateHour,84109,11146,2018-12-24 22:59:54.000,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeID,84109.0,NaN,NaN,NaN,3.591221,3.682017,2.0,2.0,2.0,5.0,27.0
IsDelivery,84109.0,NaN,NaN,NaN,0.371518,0.483213,0.0,0.0,0.0,1.0,1.0
Pax,84109.0,NaN,NaN,NaN,3.3008,3.693792,0.0,1.0,2.0,4.0,200.0
CustomerID,84109.0,NaN,NaN,NaN,12633.06576,21952.076038,0.0,0.0,0.0,16429.0,69101.0


General observations:
- There are a total of 11,147 documents (invoices)
- The maximum number of rows in a document is 46
- There are 255 different products, from 27 families
- The most frequent product is "Mineral water 1.5 Lt" which is present in 7,061 rows

Considerations on data quality:<br>
- *InvoiceDateHour* and *CustomerSince* are date/time variables, but were not recognize as such
- *CustomerID* is being recognized has numeric, but being ID is categorical
- *CustomerCity* and *CustomerSince* have a high proportion of missing values
- At a first glance, *Qty*, *TotalAmount*, and *Pax* seem to have outliers
- *ProductDesignation*, *ProductFamily*, and *InvoiceDateHour* have a high number of categories/levels

In [6]:
# Reformat variable types

cols = ['InvoiceDateHour','CustomerSince']
ds[cols] = ds[cols].apply(pd.to_datetime)

cols = ['DocNumber','ProductDesignation','CustomerID']
ds[cols] = ds[cols].astype(str)

In [7]:
# Check the differences in the summary statistics for all variables
ds.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
DocNumber,84109,11147,TK0110053522018,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductDesignation,84109,255,MINERAL WATER 1.5LT,7061,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductFamily,84109,27,STARTERS,14148,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Qty,84109.0,NaN,NaN,NaN,1.261256,1.0,1.0,1.0,1.0,53.0,0.92378
TotalAmount,84109.0,NaN,NaN,NaN,9.83142,0.0,3.0,6.9,12.6,3000.0,20.656789
InvoiceDateHour,84109,NaN,NaN,NaN,2018-07-05 03:57:12.328038656,2018-01-01 19:12:12,2018-03-23 22:48:58,2018-07-04 22:57:16,2018-10-13 22:51:24,2018-12-31 22:45:17,NaN
EmployeeID,84109.0,NaN,NaN,NaN,3.591221,2.0,2.0,2.0,5.0,27.0,3.682017
IsDelivery,84109.0,NaN,NaN,NaN,0.371518,0.0,0.0,0.0,1.0,1.0,0.483213
Pax,84109.0,NaN,NaN,NaN,3.3008,0.0,1.0,2.0,4.0,200.0,3.693792
CustomerID,84109,2316,0,52861,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Check products cardinality
ds['ProductDesignation'].value_counts()

ProductDesignation
MINERAL WATER 1.5LT      7061
EGG FRIED RICE           4604
SPRING ROLL              4079
DELIVERY CHARGE          3892
SWEET SOUR CHICKEN       3616
                         ... 
CLASSIC ROLLS VEG           1
BABATZIM ERODIOS ROSE       1
VEUVE CLICQUOT              1
HADJIMICHALI LEFKOS         1
 NO BABY CORN               1
Name: count, Length: 255, dtype: int64

Some "products" are not products, like the "DELIVERY CHARGE". Others were only sold once. These products could be deleted from the analysis.

## Data preparation

In [9]:
# Replace missepelled cities names with the correct ones

# LAKATAMEIA
ds['CustomerCity'] = np.where(ds['CustomerCity']=='LAKSTAMEIA','LAKATAMEIA',ds['CustomerCity'])
ds['CustomerCity'] = np.where(ds['CustomerCity']=='LAKATAME','LAKATAMEIA',ds['CustomerCity'])

# STROBOLOS
ds['CustomerCity'] = np.where(ds['CustomerCity']=='STROBOLO','STROBOLOS',ds['CustomerCity'])

# EGKOMI
ds['CustomerCity'] = ds['CustomerCity'].str.strip()

In [10]:
def removeOutliers(df, colList, lowPercentile=0.05, highPercentile=0.95, verbose=False):
    """Remove rows with outliers below or above the predefined percentile thresholds

    Args:
        df (dataframe): dataframe object
        colList (string list): list with names of columns
        lowPercentile (float, optional): Rows with value below this threshold will be removed. Defaults to 0.05.
        highPercentile (float, optional): Rows with value above this threshold will be removed. Defaults to 0.95.
        verbose (bool, optional): Indication if it should give feedback on the percentiles. Defaults to False.

    Returns:
        dataframe: the dataframe with the rows removed.
    """
    # Identify percentiles
    quant_df = df[colList].quantile([lowPercentile, highPercentile])
    if verbose:
        print(quant_df)

    # Loop in each column
    for name in list(df[colList].columns):

        # Keep only rows that are inside the limits
        df = df[(df[name] >= quant_df.loc[lowPercentile, name]) & (df[name] <= quant_df.loc[highPercentile, name])]

    return df

In [11]:
# Remove outliers
cols = ['Qty','TotalAmount','Pax']
ds = removeOutliers(ds, colList=cols , lowPercentile=0.00, highPercentile=0.99, verbose=True)

      Qty  TotalAmount   Pax
0.00  1.0          0.0   0.0
0.99  5.0         60.0  14.0


In [12]:
# Remove not useful products
notUsefulProducts = ['MINERAL WATER 1.5LT','DELIVERY CHARGE']
ds = ds[~ds['ProductDesignation'].isin(notUsefulProducts)]

In [13]:
# Remove products that start with " Extra ", " No ", or " With " as they are modifiers and not "real products"
prefixList = [' EXTRA ',' NO ', ' WITH ']
for p in prefixList:
    ds = ds[~ds['ProductDesignation'].str.startswith(p)]

In [14]:
# Remove sales of transiant customers
ds = ds[pd.to_numeric(ds['CustomerID'])>0]

<br><br><br><br><br>
## "Did you forget problem"

In [15]:
# From the customers with more purchases, let's select one
topCustomers = ds.groupby('CustomerID')['DocNumber'].nunique().sort_values(ascending=False)
topCustomers.head(20)

CustomerID
1567     42
58239    36
67875    30
53159    29
300      28
44685    27
68073    21
67025    21
68002    19
62241    17
30887    17
29181    16
1032     16
45121    16
1583     16
14155    15
1533     15
61587    15
44841    14
57173    14
Name: DocNumber, dtype: int64

In [16]:
# Check the purchases of the customer
cID = '1533'
ds[ds['CustomerID']==cID]

,DocNumber,ProductDesignation,ProductFamily,Qty,TotalAmount,InvoiceDateHour,EmployeeID,IsDelivery,Pax,CustomerID,CustomerCity,CustomerSince
13394,TKD0320007912018,NOODLES BEAN SPROUTS,RICE,1.0,6.7,2018-02-22 19:04:22,6,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
13395,TKD0320007912018,CUCUMBER MAKI,JAP SUSHI,1.0,3.2,2018-02-22 19:04:22,6,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
13396,TKD0320007912018,AVOCADO MAKI,JAP SUSHI,1.0,3.2,2018-02-22 19:04:22,6,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
13397,TKD0320007912018,JIRA PULAO,RICE IND,1.0,6.2,2018-02-22 19:04:22,6,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
13398,TKD0320007912018,CHICK TIKKA MASALA,MAIN IND,1.0,11.5,2018-02-22 19:04:22,6,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
...,...,...,...,...,...,...,...,...,...,...,...,...
80834,TKD0320042872018,JIRA PULAO,RICE IND,3.0,18.6,2018-12-22 20:44:10,2,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
80835,TKD0320042872018,NAAN,BREADS IND,1.0,3.2,2018-12-22 20:44:10,2,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
80836,TKD0320042872018,NAAN,BREADS IND,1.0,3.2,2018-12-22 20:44:10,2,1,1,1533,EGKOMI,2016-06-22 20:13:33.750
80837,TKD0320042872018,POPPDUMS,SIDE IND,1.0,4.9,2018-12-22 20:44:10,2,1,1,1533,EGKOMI,2016-06-22 20:13:33.750


In [17]:
# Create a pivot table with products per document
ptDocumentProduct = pd.pivot_table(ds[ds['CustomerID']==cID][['DocNumber','ProductDesignation']],
                    index='DocNumber',                             # Each row will be a document (transaction)
                    columns='ProductDesignation',                  # Each product will be a column
                    aggfunc=lambda x: 1 if len(x)>0 else 0).fillna(0)
                            
print(ptDocumentProduct.shape)
ptDocumentProduct.head()

(15, 32)


ProductDesignation,1/2 DUCK,ALOO BAINGAN,AVOCADO MAKI,BEEF BBS,CALIFORNIA MAKI,CAULIFLOWER GUNCHAO,CHICK TIKKA MASALA,CHICKEN VINDALOO,CUCUMBER MAKI,DUO TANDOORI FLOWERS,...,SPRING ROLL,STEAMED RICE,STEAMED RICE IND,SWEET SOUR CHICKEN,TANDOORI TILL JHEENGA,TEMPURA MAKI,TSANTA,VEG BBS,VEG RICE,YELLOW DAL DHABA
DocNumber,,,,,,,,,,,,,,,,,,,,,
TKD0320007912018,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0320011972018,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0320012532018,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0320012842018,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0320015772018,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Compute the distance and transform it to a dataframe for visualization
# Info on the "dice" implementation https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.dice.html
baskets = ptDocumentProduct.to_numpy().T
distMatrix = pairwise_distances(baskets,metric='dice')
distMatrixDF = pd.DataFrame(distMatrix, columns=ptDocumentProduct.columns, index=ptDocumentProduct.columns)
distMatrixDF = distMatrixDF.apply(lambda x: 1-x, axis=1) # Transform dissimilarity to similarity
distMatrixDF

/Users/nunoantonio/opt/anaconda3/envs/Python310/lib/python3.10/site-packages/sklearn/metrics/pairwise.py:2025: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)


ProductDesignation,1/2 DUCK,ALOO BAINGAN,AVOCADO MAKI,BEEF BBS,CALIFORNIA MAKI,CAULIFLOWER GUNCHAO,CHICK TIKKA MASALA,CHICKEN VINDALOO,CUCUMBER MAKI,DUO TANDOORI FLOWERS,...,SPRING ROLL,STEAMED RICE,STEAMED RICE IND,SWEET SOUR CHICKEN,TANDOORI TILL JHEENGA,TEMPURA MAKI,TSANTA,VEG BBS,VEG RICE,YELLOW DAL DHABA
ProductDesignation,,,,,,,,,,,,,,,,,,,,,
1/2 DUCK,1.000000,0.000000,0.142857,0.666667,0.333333,0.000000,0.000000,0.000000,0.250000,0.000000,...,0.500000,1.000000,0.000000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000
ALOO BAINGAN,0.000000,1.000000,0.000000,0.000000,0.000000,0.666667,0.200000,1.000000,0.000000,1.000000,...,0.000000,0.000000,0.500000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
AVOCADO MAKI,0.142857,0.000000,1.000000,0.266667,0.555556,0.133333,0.636364,0.000000,0.700000,0.000000,...,0.375000,0.142857,0.250000,0.142857,0.000000,0.142857,0.444444,0.142857,0.266667,0.000000
BEEF BBS,0.666667,0.000000,0.266667,1.000000,0.285714,0.000000,0.000000,0.000000,0.444444,0.000000,...,0.800000,0.666667,0.000000,0.666667,0.000000,0.000000,0.285714,0.000000,0.000000,0.000000
CALIFORNIA MAKI,0.333333,0.000000,0.555556,0.285714,1.000000,0.000000,0.142857,0.000000,0.666667,0.000000,...,0.250000,0.333333,0.000000,0.000000,0.000000,0.333333,0.400000,0.333333,0.285714,0.000000
CAULIFLOWER GUNCHAO,0.000000,0.666667,0.133333,0.000000,0.000000,1.000000,0.363636,0.666667,0.000000,0.666667,...,0.000000,0.000000,0.400000,0.000000,0.666667,0.000000,0.000000,0.000000,0.000000,0.666667
CHICK TIKKA MASALA,0.000000,0.200000,0.636364,0.000000,0.142857,0.363636,1.000000,0.200000,0.375000,0.200000,...,0.000000,0.000000,0.500000,0.000000,0.200000,0.000000,0.285714,0.000000,0.181818,0.200000
CHICKEN VINDALOO,0.000000,1.000000,0.000000,0.000000,0.000000,0.666667,0.200000,1.000000,0.000000,1.000000,...,0.000000,0.000000,0.500000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
CUCUMBER MAKI,0.250000,0.000000,0.700000,0.444444,0.666667,0.000000,0.375000,0.000000,1.000000,0.000000,...,0.400000,0.250000,0.200000,0.250000,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000


In [19]:
# Let's define a the current basket to simulate the problem
currentBasket = ['BEEF BBS','SPICY SALMON MAKI','AVOCADO MAKI','TEMPURA MAKI']

In [20]:
# Let's create a list of products not in the current basket
itemsNotInBasket = [x for x in distMatrixDF.columns if x not in currentBasket]

# Create a temporary dataframe with rows in basket and columns not in basket
tempDF = distMatrixDF[distMatrixDF.index.isin(currentBasket)][itemsNotInBasket]

# Create a list to store the results
results_list = []

# For each item not in the basket
for c in itemsNotInBasket:
    # Get maximum similarity value of the item (index of the row of the item in the basket)
    indexOfMax = tempDF[c].idxmax()
    # Add the results to the list
    results_list.append({
        'NotInBasket': c,
        'InBasket': tempDF.loc[indexOfMax].name,
        'Similarity': tempDF.loc[indexOfMax, c]
    })

# Create a DataFrame from the list
maxSimilarDF = pd.DataFrame(results_list)

# Sort results by similarity
maxSimilarDF = maxSimilarDF.sort_values(by='Similarity', ascending=False)

In [21]:
# Simulate presentation of top n recommendations (Recommendations are the items "NotInBaskket")
n = 5
maxSimilarDF.head(n)

,NotInBasket,InBasket,Similarity
25,VEG BBS,TEMPURA MAKI,1.00
14,PRAWN CRACKERS,BEEF BBS,0.80
19,SPRING ROLL,BEEF BBS,0.80
2,CALIFORNIA MAKI,SPICY SALMON MAKI,0.75
6,CUCUMBER MAKI,AVOCADO MAKI,0.70


### **EXERCISE 1**
Check how results change:

- Using different measures (check the different measures in https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html)
- Changing the items in the "currentBasket"
- With other customers

<br><br><br><br><br>
## Recommendation based on all previous purchases (recommend a drink to a dish)

In [22]:
# List all the product families
ds['ProductFamily'].value_counts()

ProductFamily
STARTERS      6635
RICE          4874
MEAT          3578
SIZZLING      1889
JAP SUSHI     1863
TSANTES       1797
MAIN IND      1122
BREADS IND     759
START IND      675
RICE IND       643
VEG IND        410
SOUPS          328
SEAFOOD        311
DRINKS         238
VEG            188
NEW SUSHI      152
SIDE IND       125
SPIRITS          4
EXTRAS           2
Name: count, dtype: int64

In [23]:
# Let's define the current basket has one food product to simulate the problem
currentBasket = ['1/4 DUCK']

# Families to recomend
familiesToRecommend = ['DRINKS']

In [24]:
# Create a pivot table with products per document (only desired products)
ptDocumentProduct = pd.pivot_table(ds[(ds['ProductDesignation'].isin(currentBasket)) | (ds['ProductFamily'].isin(familiesToRecommend))][['DocNumber','ProductDesignation']],
                    index='DocNumber',                             # Each row will be a document (transaction)
                    columns='ProductDesignation',                  # Each product will be a column
                    aggfunc=lambda x: 1 if len(x)>0 else 0).fillna(0)
                            
print(ptDocumentProduct.shape)
ptDocumentProduct.head()

(1302, 11)


ProductDesignation,1/4 DUCK,7UP,COKE,COKE ZERO,DIET 7UP,DIET COKE,DIET SPRITE,PERRIER,SODA,SPRITE,WINE
DocNumber,,,,,,,,,,,
TKD0310000022018,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0310000042018,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0310000072018,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0310000112018,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TKD0310000132018,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# Compute the distance and transform it to a dataframe for visualization
baskets = ptDocumentProduct.to_numpy().T
distMatrix = pairwise_distances(baskets,metric='dice')
distMatrixDF = pd.DataFrame(distMatrix, columns=ptDocumentProduct.columns, index=ptDocumentProduct.columns)
distMatrixDF = distMatrixDF.apply(lambda x: 1-x, axis=1) # Transform dissimilarity to similarity
distMatrixDF

/Users/nunoantonio/opt/anaconda3/envs/Python310/lib/python3.10/site-packages/sklearn/metrics/pairwise.py:2025: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)


ProductDesignation,1/4 DUCK,7UP,COKE,COKE ZERO,DIET 7UP,DIET COKE,DIET SPRITE,PERRIER,SODA,SPRITE,WINE
ProductDesignation,,,,,,,,,,,
1/4 DUCK,1.000000,0.0,0.024430,0.011466,0.000000,0.005141,0.003448,0.0,0.006838,0.011956,0.0
7UP,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
COKE,0.024430,0.0,1.000000,0.096552,0.024691,0.021978,0.071429,0.0,0.085106,0.063158,0.0
COKE ZERO,0.011466,0.0,0.096552,1.000000,0.108108,0.095238,0.051948,0.0,0.022989,0.090909,0.0
DIET 7UP,0.000000,0.0,0.024691,0.108108,1.000000,0.100000,0.000000,0.0,0.000000,0.000000,0.0
DIET COKE,0.005141,0.0,0.021978,0.095238,0.100000,1.000000,0.000000,0.0,0.000000,0.058824,0.0
DIET SPRITE,0.003448,0.0,0.071429,0.051948,0.000000,0.000000,1.000000,0.0,0.000000,0.000000,0.0
PERRIER,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.0
SODA,0.006838,0.0,0.085106,0.022989,0.000000,0.000000,0.000000,0.0,1.000000,0.054054,0.0


In [26]:
# Let's create a list of products not in the current basket
itemsNotInBasket = [x for x in distMatrixDF.columns if x not in currentBasket]

# Create a temporary dataframe with rows in basket and columns not in basket and 
tempDF = distMatrixDF[distMatrixDF.index.isin(currentBasket)][itemsNotInBasket]

# Create a max similarity dataframe do store the results
maxSimilarDF = pd.DataFrame(columns=['NotInBasket','InBasket','Similarity'])

# For each item not in the basket
for c in itemsNotInBasket:
    # Get maximum similarity value of the item (index of the row of the item in the basket)
    indexOfMax = tempDF[c].argmax()
    # Add the results to the dataframe
    new_row = pd.DataFrame({'NotInBasket':[c],
                    'InBasket':[tempDF.iloc[indexOfMax].name],
                    'Similarity':[tempDF.iloc[indexOfMax][c]]
                    })
    maxSimilarDF = pd.concat([maxSimilarDF,new_row],ignore_index=True)

# Sort results by similarity
maxSimilarDF = maxSimilarDF.sort_values(by='Similarity', ascending=False)

In [27]:
# Simulate presentation of top n recommendations (Recommendations are the items "NotInBaskket")
n = 5
maxSimilarDF.head(n)

,NotInBasket,InBasket,Similarity
1,COKE,1/4 DUCK,0.024430
8,SPRITE,1/4 DUCK,0.011956
2,COKE ZERO,1/4 DUCK,0.011466
7,SODA,1/4 DUCK,0.006838
4,DIET COKE,1/4 DUCK,0.005141


### **EXERCISE 2**
Check how results change:

- Using different measures (check the different measures in https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html)
- Changing the items in the "currentBasket"